# Create and Review Metadata Files

This notebook prepares `metadata.json`. It never writes while cells are merely executed. First inspect the exact before/after comparison, then choose **Confirm overwrite** or **Reject changes**.

`metadata.json` describes the original measurement and its preprocessing. New Module 13 reuse metadata is created later by Lab 13 as `metadata_reused.json`.

<div style="border: 3px solid #b45309; background: #fff7ed; padding: 18px 20px; margin: 16px 0 24px 0; border-radius: 8px;">
<h2 style="margin-top: 0; color: #9a3412;">Important: Metadata are not normally generated all at once</h2>
<p>In a real research workflow, metadata come from different sources. Some metadata are <strong>recorded automatically</strong> by instruments or software, some are <strong>extracted from the recorded files</strong>, and some must be <strong>entered or reviewed manually</strong> by the researchers who understand the experiment.</p>
<p>In the JupyterHub environment used for this course, JSON files cannot be edited manually. This notebook therefore provides a structured way to review, supplement, and write the metadata. It does not mean that all metadata should normally be invented or generated automatically.</p>
<p><strong>Your task is still to check where each value comes from and whether it correctly describes your measurement.</strong></p>
</div>

## Section 0: Choose the Preparation Mode

| Mode | Behaviour |
|---|---|
| `'load'` | Load and validate the existing file. No write buttons are shown. |
| `'replace'` | Prepare a complete replacement using central defaults and the selected use case. |
| `'update'` | Prepare an update while retaining the unselected use case. |

Modes `'replace'` and `'update'` only prepare a candidate. Writing requires a later button click.

In [ ]:
# Section 0: Metadata preparation mode
metadata_mode = 'load'   # 'load' | 'replace' | 'update'

valid_modes = {
    'load': 'Load and validate only',
    'replace': 'Prepare complete replacement',
    'update': 'Prepare update of selected use case',
}
if metadata_mode not in valid_modes:
    raise ValueError(f"metadata_mode must be one of {sorted(valid_modes)}")
print(valid_modes[metadata_mode])

## Section 1: Load Helpers and Existing File

The validation and write logic lives in `src/`; the notebook only supplies values and displays the result.

In [ ]:
# Section 1: Imports and current file contents
from copy import deepcopy
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import HTML, display

project_root = Path.cwd()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from metadata_loader import (
    default_public_metadata,
    load_json_file,
    load_public_metadata,
    merge_metadata_updates,
    public_metadata_path,
)
from metadata_validation import (
    ALLOWED_MEASUREMENTS,
    create_metadata_write_controls,
    metadata_diff_html,
    prepare_public_metadata_candidate,
    validate_public_metadata,
)

metadata_file = public_metadata_path(project_root)
public_before_raw = load_json_file(metadata_file, {})
existing_public_metadata = load_public_metadata(project_root)

if metadata_file.is_file():
    print(f'Loaded existing {metadata_file.name}. No file has been changed.')
else:
    print(f'WARNING: {metadata_file.name} does not exist yet.')
    print("Central defaults are shown instead. Use mode 'replace' to create the file.")

## Section 2: Dataset and General Metadata

Every input starts as `None`, which means "keep the current value". Only values you explicitly set here overwrite the base: the existing file in mode `'update'`, the central defaults in mode `'replace'`. In mode `'load'` these inputs are ignored. `measurement_type` defines the use case and must match `quantity`.

In [ ]:
# Section 2: Edit general public metadata
# Leave a value at None to keep the base value. Only explicitly set values
# overwrite the base metadata.
recorded_data_path = None   # e.g. 'data/suspension/Example/Beschleunigung ohne g.xls'
measurement_type = None     # 'drivetrain' or 'suspension'
run_name = None
quantity = None             # derived from measurement_type unless set explicitly
data_stage = None           # e.g. 'raw'
version = None              # e.g. 'v0.1.0'
hot_storage_path = None

if measurement_type is not None and quantity is None:
    quantity = ALLOWED_MEASUREMENTS[measurement_type]

base_metadata = (
    default_public_metadata() if metadata_mode == 'replace'
    else existing_public_metadata
)
general_inputs = {
    'recorded_data_path': recorded_data_path,
    'measurement_type': measurement_type,
    'run_name': run_name,
    'quantity': quantity,
    'data_stage': data_stage,
    'version': version,
    'hot_storage_path': hot_storage_path,
}
effective_general = {
    key: (value if value is not None else base_metadata.get(key))
    for key, value in general_inputs.items()
}

for key, value in effective_general.items():
    origin = 'set here' if general_inputs[key] is not None else 'kept from base'
    print(f'{key}: {value!r} ({origin})')
print('Analysis key:', f"{effective_general['measurement_type']}_{effective_general['quantity']}")

## Section 3: Selected Use-Case Parameters

Only the selected `measurement_type` is relevant here. Like in Section 2, enter **only the keys you want to change** into the two override dictionaries in the code; every other value keeps the base value (existing file in mode `'update'`, central defaults in mode `'replace'`). The cell prints the resulting parameters so you can check the effect before building the preview in Section 4.

In [ ]:
# Section 3: Selected use-case parameters
# Enter only the keys you want to change; everything else keeps the base
# value. Nested dictionaries are merged key by key.
analysis_overrides = {
    # example: 'minimum_bright_phase_mean_lx': 600,
}
setup_overrides = {
    # example: 'sensor_position': 'front axle',
}

central_defaults = default_public_metadata()
active_measurement_type = effective_general['measurement_type']
active_quantity = effective_general['quantity']
active_analysis_key = f'{active_measurement_type}_{active_quantity}'
if active_measurement_type not in ('drivetrain', 'suspension'):
    raise ValueError("measurement_type must be 'drivetrain' or 'suspension'")

active_analysis_metadata = deepcopy(
    base_metadata.get('analysis', {}).get(active_analysis_key)
    or central_defaults['analysis'][active_analysis_key]
)
active_setup_metadata = deepcopy(
    base_metadata.get(active_measurement_type)
    or central_defaults[active_measurement_type]
)

edited_analysis_metadata = merge_metadata_updates(active_analysis_metadata, analysis_overrides)
edited_setup_metadata = merge_metadata_updates(active_setup_metadata, setup_overrides)

display(HTML(f'<h3>{active_measurement_type.title()}: analysis parameters</h3>'))
print(json.dumps(edited_analysis_metadata, indent=2))
display(HTML(f'<h3>{active_measurement_type.title()}: measurement setup</h3>'))
print(json.dumps(edited_setup_metadata, indent=2))

## Section 4: Validate and Preview Before/After

Run this cell after editing Sections 2 and 3. Validation errors disable the confirmation button; warnings remain confirmable. In modes `'replace'` and `'update'`, every changed path is shown with its previous and proposed value. In mode `'load'`, the current file contents are shown instead, because nothing will be written.

In [ ]:
# Section 4: Build candidate, validate, and show exact differences
public_candidate = prepare_public_metadata_candidate(
    existing_public_metadata,
    metadata_mode,
    general_inputs,
    edited_analysis_metadata,
    edited_setup_metadata,
)
public_validation = validate_public_metadata(public_candidate, project_root)
validation_rows = [
    {'level': 'error', **item}
    for item in public_validation['errors']
] + [
    {'level': 'warning', **item}
    for item in public_validation['warnings']
]
if validation_rows:
    display(pd.DataFrame(validation_rows))
else:
    display(HTML("<div style='color:#1f7a3a;font-weight:600'>Validation passed without warnings.</div>"))

if metadata_mode == 'load':
    display(HTML('<h4>Current metadata.json (nothing will be written)</h4>'))
    print(json.dumps(public_before_raw, indent=2))
else:
    display(HTML(metadata_diff_html(public_before_raw, public_candidate, 'metadata.json: before / after')))

## Section 5: Confirm or Reject

The red button performs the overwrite. The green button explicitly rejects the candidate. If this notebook is run with **Run All**, execution continues without writing while the decision remains pending.

**After changing any value in Section 2 or 3, use "Restart" and "Run All" before confirming.** The buttons write exactly the candidate previewed in Section 4; running cells out of order can otherwise write an outdated candidate. Run All is safe here - nothing is written without a button click.

In [ ]:
# Section 5: Explicit write decision
if 'metadata_write_widget' in globals():
    metadata_write_widget.close()

if metadata_mode == 'load':
    metadata_write_state = {'decision': 'load_only'}
    display(HTML("<div style='color:#1f7a3a;font-weight:600'>Mode 'load': metadata.json will not be written.</div>"))
else:
    metadata_write_widget, metadata_write_state = create_metadata_write_controls(
        project_root,
        public_before_raw,
        public_candidate,
    )
    display(metadata_write_widget)

## Section 6: Verify the Decision

After clicking a button, run this cell again. It clearly distinguishes written, rejected, pending, and load-only states.

In [ ]:
# Section 6: Verify current on-disk state
decision = metadata_write_state.get('decision')
if decision in ('written', 'load_only'):
    verified_metadata = load_public_metadata(project_root)
    verification = validate_public_metadata(verified_metadata, project_root)
    print('Decision:', decision)
    print('metadata.json valid:', verification['valid'])
    print('Selected data exists:', (project_root / verified_metadata['recorded_data_path']).is_file())
elif decision == 'rejected':
    print('Changes were rejected. No file was written.')
elif decision == 'pending':
    print('Waiting for your button decision. No file has been written.')
else:
    print('Decision state:', decision, '- rebuild the preview before writing.')